## Part 3: Structure Prediction & Visualization

Reproduces **Figure f** from the Nature paper.  
We fold two peptide-receptor complexes for PDB entry **6VME**:
- **Blue** — true test peptide  
- **Red** — PepMLM-generated peptide  
- **Gray** — receptor (shared)

Tool: ESMFold (local, via HuggingFace transformers) → `.pdb` → py3Dmol

### 3.1 — Imports

In [18]:
import pandas as pd
import torch
import py3Dmol
from transformers import AutoTokenizer, AutoModelForMaskedLM, EsmForProteinFolding

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


### 3.2 — Load Target Entry (6VME)

In [19]:
test_df = pd.read_csv("../data/pepnn_test_dataset.csv")
entry = test_df[test_df["PDB"] == "6VME"].iloc[0]

RECEPTOR = entry["Receptor Sequence"]
TRUE_PEP = entry["Sequence"]
PEP_LEN  = len(TRUE_PEP)

print(f"PDB         : 6VME")
print(f"Receptor    : {RECEPTOR}")
print(f"True peptide: {TRUE_PEP}  (len={PEP_LEN})")
print(f"Total length: {len(RECEPTOR) + PEP_LEN} aa")

PDB         : 6VME
Receptor    : QSENNDIDEVIIPTAPLYKQILNLYAEENAIEDTIFYLGEALRRGVIDLDVFLKHVRLLSRKQFQLRALMQKARKTAGLSD
True peptide: NASSLYGISAMDGVPFTLH  (len=19)
Total length: 100 aa


### 3.3 — Generate Peptide with PepMLM

In [20]:
PEP_MODEL     = "TianlaiChen/PepMLM-650M"
pep_tokenizer = AutoTokenizer.from_pretrained(PEP_MODEL)
pep_model     = AutoModelForMaskedLM.from_pretrained(PEP_MODEL).to(device)
pep_model.eval()

def generate_peptide(receptor_seq: str, pep_len: int) -> str:
    masked_input = receptor_seq + pep_tokenizer.mask_token * pep_len
    inputs = pep_tokenizer(masked_input, return_tensors="pt",
                           truncation=True, max_length=512).to(device)
    mask_positions = (inputs["input_ids"] == pep_tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
    with torch.no_grad():
        logits = pep_model(**inputs).logits
    predicted_ids = logits[0, mask_positions].argmax(dim=-1)
    return pep_tokenizer.decode(predicted_ids, skip_special_tokens=True).replace(" ", "")

GEN_PEP = generate_peptide(RECEPTOR, PEP_LEN)
print(f"Generated peptide: {GEN_PEP}")
print(f"True peptide     : {TRUE_PEP}")

Loading weights: 100%|██████████| 540/540 [00:00<00:00, 53577.24it/s]


Generated peptide: NASSLYGISSMDVVVFSAH
True peptide     : NASSLYGISAMDGVPFTLH


### 3.4 — Load ESMFold & Fold Both Complexes

ESMFold runs locally — no API or token needed. First download is ~2.7 GB.  
For a 100 aa sequence, folding takes ~1-3 min on CPU.

In [21]:
import time

print("Step 1/4: Loading ESMFold tokenizer...")
fold_tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
print("Step 2/4: Loading ESMFold model (~2.7 GB, may take a few minutes)...")
fold_model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1",
                                                   low_cpu_mem_usage=True)
fold_model = fold_model.to(device)
fold_model.eval()
print("ESMFold loaded.")

def fold_sequence(sequence: str) -> str:
    inputs = fold_tokenizer(sequence, return_tensors="pt",
                            add_special_tokens=False).to(device)
    with torch.no_grad():
        output = fold_model(**inputs)
    return fold_model.output_to_pdb(output)[0]

print("Step 3/4: Folding true complex...")
t0 = time.time()
pdb_true = fold_sequence(RECEPTOR + TRUE_PEP)
print(f"Done in {time.time()-t0:.1f}s")

print("Step 4/4: Folding generated complex...")
t0 = time.time()
pdb_gen = fold_sequence(RECEPTOR + GEN_PEP)
print(f"Done in {time.time()-t0:.1f}s")

Step 1/4: Loading ESMFold tokenizer...
Step 2/4: Loading ESMFold model (~2.7 GB, may take a few minutes)...


Loading weights: 100%|██████████| 4498/4498 [00:46<00:00, 96.87it/s]  
[transformers] EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status  | 
-----------------------------------+---------+-
esm.contact_head.regression.weight | MISSING | 
esm.contact_head.regression.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ESMFold loaded.
Step 3/4: Folding true complex...
Done in 217.9s
Step 4/4: Folding generated complex...
Done in 206.8s


### 3.5 — Visualize Structures

Each viewer shows the receptor (gray) with the peptide highlighted.  
- **Left:** True test peptide (blue) — paper's reference  
- **Right:** PepMLM-generated peptide (red) — our replication

In [23]:
def make_viewer(pdb_string: str, receptor_len: int, peptide_color: str, label: str):
    view = py3Dmol.view(width=500, height=450)
    view.addModel(pdb_string, "pdb")
    view.setStyle(
        {"resi": f"1-{receptor_len}"},
        {"cartoon": {"color": "lightgray", "opacity": 0.85}}
    )
    view.setStyle(
        {"resi": f"{receptor_len + 1}-{receptor_len + 100}"},
        {"cartoon": {"color": peptide_color, "thickness": 0.6}}
    )
    view.zoomTo({"resi": f"{receptor_len + 1}-{receptor_len + 100}"})
    view.setBackgroundColor("white")
    view.addLabel(label, {"fontSize": 14, "fontColor": "black",
                          "backgroundOpacity": 0,
                          "position": {"x": 0, "y": 0, "z": 0}})
    return view

receptor_len = len(RECEPTOR)

print("=== True Peptide (Blue) ===")
view_true = make_viewer(pdb_true, receptor_len, "blue", f"6VME - True: {TRUE_PEP}")
view_true.show()

print("=== Generated Peptide (Red) ===")
view_gen = make_viewer(pdb_gen, receptor_len, "red", f"6VME - Generated: {GEN_PEP}")
view_gen.show()

=== True Peptide (Blue) ===


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

=== Generated Peptide (Red) ===


3Dmol.js failed to load for some reason. Please check your browser console for error messages.